# Fine-tuning a Frontier Model

Based on Week 6 Day 5: Fine-tune\ing GPT-4.1-nano on the Amazon pricing dataset.

In [ ]:
import os
import sys
import re
import json
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

sys.path.insert(0, str(Path.cwd().parents[1]))
from pricer.items import Item
from pricer.evaluator import evaluate

In [ ]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

openai = OpenAI()

In [ ]:
LITE_MODE = True
TRAIN_SIZE = 1000
VAL_SIZE = 100
# MODEL = "gpt-4.1-nano-2025-04-14"
MODEL = "ft:gpt-4.1-nano-2025-04-14:croissant-inc:pricer:DHAXMjSo"
N_EPOCHS = 3
BATCH_SIZE = 4
SUFFIX = "pricer"
SEED = 42

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
fine_tune_train = train[:TRAIN_SIZE]
fine_tune_validation = val[:VAL_SIZE]

print(f"Fine-tune train: {len(fine_tune_train)}, validation: {len(fine_tune_validation)}")

In [ ]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

messages_for(fine_tune_train[0])

In [ ]:
def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str + '}\n'
    return result.strip()

print(make_jsonl(fine_tune_train[:3]))

In [ ]:
jsonl_dir = Path("jsonl")
jsonl_dir.mkdir(exist_ok=True)

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        f.write(make_jsonl(items))

write_jsonl(fine_tune_train, jsonl_dir / "fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, jsonl_dir / "fine_tune_validation.jsonl")
print("JSONL files written.")

In [ ]:
with open(jsonl_dir / "fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

with open(jsonl_dir / "fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

print(f"Train file: {train_file.id}")
print(f"Validation file: {validation_file.id}")

In [ ]:
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model=MODEL,
    seed=SEED,
    hyperparameters={"n_epochs": N_EPOCHS, "batch_size": BATCH_SIZE},
    suffix=SUFFIX,
)

job_id = job.id
print(f"Job created: {job_id}")

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id)

In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

You can also monitor your job in the OpenAI dashboard: https://platform.openai.com/finetune

Re-run the two cells above to check progress. Once the status is `succeeded`, continue below.

In [ ]:
job = openai.fine_tuning.jobs.retrieve(job_id)
if job.status == "succeeded":
    fine_tuned_model_name = job.fine_tuned_model
    print(f"Fine-tuned model: {fine_tuned_model_name}")
else:
    print(f"Job status: {job.status}")
    if job.status == "running":
        print("Re-run this cell once the job has succeeded to get the model name.")
    elif job.status == "failed" and job.error:
        print(f"Error: {job.error.message}")
    if job.fine_tuned_model:
        fine_tuned_model_name = job.fine_tuned_model
        print(f"Model: {fine_tuned_model_name}")

In [ ]:
def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

test_messages_for(test[0])

In [ ]:
def gpt_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7,
    )
    return response.choices[0].message.content

In [ ]:
print(f"Actual:    ${test[0].price:.2f}")
print(f"Predicted: {gpt_fine_tuned(test[0])}")

In [ ]:
evaluate(gpt_fine_tuned, test)

## Reference scores

| Config | Error |
|---|---|
| nano 200 examples | $96.58 |
| mini 200 examples | $96.58 |
| mini 2,000 examples | $79.29 |
| nano 2,000 examples | $82.26 |
| nano 20,000 examples | $67.75 |